# Retail Demand Forecasting with Google TimesFM 2.5 (Favorita)

This notebook is a production-style, end-to-end tutorial for SKU-store forecasting, inventory recommendation, and warehouse allocation.

## 1. Business Problem and Decision Context

A retailer operates thousands of item-store combinations. Every day the operations team must decide:

- How many units to order for each item-store pair
- Which warehouse should ship replenishment
- Which stores are likely to stock out without action

We forecast demand for **7 / 14 / 30 days**, then convert forecasts into inventory and allocation decisions.

## 2. What This Notebook Covers

1. Download Favorita data directly from the web (Kaggle API)
2. Build full-scale item-store panel data
3. Forecast with TimesFM 2.5 (core + optional XReg covariate path)
4. Evaluate against baselines with leakage-safe backtesting
5. Convert forecasts to reorder recommendations
6. Allocate stock with heuristic and MILP optimization
7. Optionally generate Kaggle submission format

> Assumption: You have accepted the Kaggle competition rules for `favorita-grocery-sales-forecasting`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from __future__ import annotations

from datetime import timedelta
from pathlib import Path

import pandas as pd
import polars as pl

from scripts.retail_demand_timesfm_favorita import (
    PipelineConfig,
    build_inventory_policy,
    build_submission,
    build_test_promo_lookup,
    ensure_paths,
    evaluate_backtest,
    forecast_horizon,
    get_anchor_date,
    get_model_and_compile,
    heuristic_allocation,
    load_metadata,
    milp_allocation,
    prepare_series_panel,
    set_global_seeds,
    download_favorita_data,
)

## 3. Environment Setup

Use `uv` for dependency management.

In [ ]:
# Run once in the project root if needed:
# !uv venv
# !uv sync --locked --group applied --group xreg
# !python -m ipykernel install --user --name timesfm-retail --display-name "Python (timesfm-retail)"

## 4. Configuration

Tune these defaults for your machine and runtime budget.

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts/retail_demand_timesfm"

cfg = PipelineConfig(
    data_root=DATA_ROOT,
    artifacts_root=ARTIFACTS_ROOT,
    horizons=(7, 14, 30),
    context_len=256,
    per_core_batch_size=8,
    forecast_batch_size=64,
    seed=42,
    device="auto",         # auto | cuda | cpu
    use_xreg=True,          # TimesFM covariate path
    xreg_mode="xreg + timesfm",
    xreg_ridge=1e-3,
    run_milp=True,
    include_submission=True,
)

paths = ensure_paths(cfg)
set_global_seeds(cfg.seed)
cfg

## 5. Download Favorita Data from the Web

This cell downloads and extracts required competition files directly from Kaggle.

In [ ]:
download_favorita_data(paths["raw"])
print("Downloaded files:")
print(sorted([p.name for p in paths["raw"].glob("*.csv")]))

## 6. Build Full-Scale Panel Dataset

We filter training rows to item-store pairs present in test and materialize:

- `train_long_filtered.parquet`
- `series_panel.parquet` (one row per series, list columns)

In [ ]:
train_long_path, panel_path = prepare_series_panel(
    raw_dir=paths["raw"],
    processed_dir=paths["processed"],
    max_series=cfg.max_series,
)

anchor = get_anchor_date(train_long_path)
print("Train long:", train_long_path)
print("Series panel:", panel_path)
print("Forecast anchor date:", anchor)

## 7. Metadata and Feature Inputs

Load item/store metadata and test promotions used by XReg covariate forecasting.

In [ ]:
items_df, stores_df, test_df = load_metadata(paths["raw"])
family_by_item = dict(zip(items_df["item_nbr"].astype(int), items_df["family"].astype(str)))
cluster_by_store = dict(zip(stores_df["store_nbr"].astype(int), stores_df["cluster"].astype(int)))
perishable_weight_by_item = dict(
    zip(
        items_df["item_nbr"].astype(int),
        (items_df["perishable"].astype(int).map({1: 1.25, 0: 1.0})).astype(float),
    )
)
test_promo_lookup = build_test_promo_lookup(test_df)

items_df.head(), stores_df.head(), test_df.head()

## 8. Load and Compile TimesFM 2.5

We compile once at `max_horizon = max(30, 16)` because we also build optional 16-day submission outputs.

In [ ]:
submission_horizon = 16 if cfg.include_submission else 0
max_horizon = max(max(cfg.horizons), submission_horizon)

model = get_model_and_compile(
    cfg=cfg,
    max_horizon=max_horizon,
    enable_backcast=cfg.use_xreg,
)

## 9. Forecast 7 / 14 / 30 Days

This is the main production inference stage.

In [ ]:
forecast_paths = {}
stats_paths = {}

for horizon in cfg.horizons:
    forecast_path, stats_path = forecast_horizon(
        model=model,
        panel_path=panel_path,
        output_dir=paths["artifacts"],
        anchor_date=anchor,
        horizon=horizon,
        cfg=cfg,
        family_by_item=family_by_item,
        cluster_by_store=cluster_by_store,
        test_promo_lookup=test_promo_lookup,
    )
    forecast_paths[horizon] = forecast_path
    stats_paths[horizon] = stats_path

forecast_paths

## 10. Validation vs Baselines (Leakage-Safe)

We run historical backtest and compare TimesFM against:

- Last-value naive
- Seasonal-7 naive

Metrics include weighted RMSLE-style score and WMAPE.

In [ ]:
metrics_path = evaluate_backtest(
    model=model,
    panel_path=panel_path,
    train_long_path=train_long_path,
    artifacts_dir=paths["artifacts"],
    cfg=cfg,
    family_by_item=family_by_item,
    cluster_by_store=cluster_by_store,
    perishable_weight_by_item=perishable_weight_by_item,
)

pd.read_csv(metrics_path)

## 11. Inventory Policy from Forecast Uncertainty

For each item-store, we compute:

- Lead-time demand estimate
- Safety stock (quantile spread aware)
- Reorder point
- Recommended order quantity

In [ ]:
h7 = min(cfg.horizons)
inventory_path = build_inventory_policy(
    forecast_h7_path=forecast_paths[h7],
    series_stats_path=stats_paths[h7],
    stores_df=stores_df,
    items_df=items_df,
    cfg=cfg,
    artifacts_dir=paths["artifacts"],
)

inv_df = pl.read_parquet(inventory_path).to_pandas()
inv_df.head()

## 12. Warehouse Allocation: Heuristic and MILP

- **Heuristic**: fast rule-based fill from primary warehouse then spillover
- **MILP**: constrained optimization for allocation cost + shortage penalty

In [ ]:
heuristic_path = heuristic_allocation(
    inventory_path=inventory_path,
    stores_df=stores_df,
    cfg=cfg,
    artifacts_dir=paths["artifacts"],
)

milp_path = None
if cfg.run_milp:
    milp_path = milp_allocation(
        inventory_path=inventory_path,
        stores_df=stores_df,
        cfg=cfg,
        artifacts_dir=paths["artifacts"],
    )

print("Heuristic allocation:", heuristic_path)
print("MILP allocation:", milp_path)

## 13. Optional Kaggle Submission (16-Day Horizon)

Generates `id,unit_sales` CSV using TimesFM forecast mapped to test horizon days.

In [ ]:
submission_path = None
if cfg.include_submission:
    f16_path, _ = forecast_horizon(
        model=model,
        panel_path=panel_path,
        output_dir=paths["artifacts"],
        anchor_date=anchor,
        horizon=16,
        cfg=cfg,
        family_by_item=family_by_item,
        cluster_by_store=cluster_by_store,
        test_promo_lookup=test_promo_lookup,
    )
    submission_path = build_submission(
        forecast_h16_path=f16_path,
        raw_dir=paths["raw"],
        anchor_date=anchor,
        artifacts_dir=paths["artifacts"],
    )

submission_path

## 14. Inspect Outputs

All artifacts are under `artifacts/retail_demand_timesfm/`.

In [ ]:
for p in sorted(paths["artifacts"].rglob("*")):
    if p.is_file():
        print(p)

## 15. Next Operationalization Steps

1. Schedule this pipeline daily and persist forecasts in your planning DB.
2. Replace proxy warehouse mapping with real network topology and transport costs.
3. Add service-level policies by product family and business criticality.
4. Track model drift and backtest metrics over time.